# **INF01017 - Aprendizado de Máquina - INF/UFRGS**
#### *Profa. Mariana Recamonde-Mendoza (mrmendoza@inf.ufrgs.br)*
<br>

## Atividade Prática: Métodos de Ensemble

---
***Observação:*** *Este notebook é disponibilizado aos alunos como complemento às aulas e aos slides preparados pela professora. Os principais conceitos são apresentados no material teórico fornecido. O objetivo deste notebook é reforçar os conceitos e demonstrar questões práticas no uso de diferentes algoritmos de ensemble.*

---

**Objetivo deste notebook:** Comparar experimentalmente quatro métodos de ensemble — *Bagging*, *Random Forests*, *AdaBoost* e *Gradient Boosting* — entre si e com uma árvore de decisão simples (modelo de referência), utilizando o dataset **Optical Recognition of Handwritten Digits** (digits). Ao longo dos experimentos, você irá:
1. Aplicar boas práticas de divisão de dados e evitar *data leakage* com **validação cruzada aninhada** (*nested cross-validation*).
2. Otimizar o hiperparâmetro `n_estimators` (número de modelos no ensemble) via busca em grade com **3 folds internos**, obtendo uma estimativa de desempenho não viesada com **3 folds externos**.
3. Analisar como métodos paralelos (Bagging, Random Forests) beneficiam modelos de **alta variância**, e métodos sequenciais (AdaBoost, Gradient Boosting) beneficiam modelos de **alto viés**.
4. Comparar o desempenho final dos cinco modelos no conjunto de teste holdout.


---
# Parte 0 — Conceitos Fundamentais

## O que são métodos de ensemble?

Métodos de ensemble combinam múltiplos modelos base (*base learners*) para produzir uma predição conjunta mais robusta e precisa do que qualquer modelo individual. A ideia central é que **erros cometidos por diferentes modelos tendem a se cancelar** quando as predições são agregadas.

## Viés e Variância

O desempenho de generalização de um modelo pode ser decomposto em três componentes: Viés, variância e ruído irredutível.

| Componente | O que representa | Sintoma |
|---|---|---|
| **Viés** | Erro sistemático (modelo muito simples) | *Underfitting*: erra no treino e no teste |
| **Variância** | Sensibilidade ao ruído nos dados (modelo muito complexo) | *Overfitting*: acerta no treino, erra no teste |


O Ruído irredutível é a parcela do erro que nenhum modelo consegue eliminar, pois decorre de incerteza intrínseca ao próprio problema — features não observadas, imprecisão na medição do alvo ou variabilidade genuína do fenômeno. Ele estabelece um teto de desempenho: reduzir viés e variância ao mínimo possível ainda deixa esse resíduo. A única forma de diminuí-lo é melhorar os dados (coletar features mais informativas ou medir o alvo com mais precisão), nunca o modelo.



## Famílias de Ensemble

### Métodos Paralelos — reduzem a **variância**
Treinam os modelos base de forma **independente** (em paralelo), em geral com amostras aleatórias do conjunto de treino (*bootstrap*). São especialmente úteis quando o modelo base tem alta variância (ex.: árvores de decisão profundas, sem poda).

- **Bagging** (*Bootstrap AGGregatiNG*): cada modelo base é treinado em uma amostra com reposição (*bootstrap*) do treino. As predições finais são agregadas por votação majoritária.
- **Random Forests**: variante do Bagging onde, além do bootstrap nas amostras, uma seleção aleatória de features é considerada em cada divisão da árvore, aumentando a diversidade entre os modelos.

### Métodos Sequenciais — reduzem o **viés**
Treinam os modelos base **sequencialmente**, onde cada novo modelo foca nos erros cometidos pelo anterior. São especialmente úteis quando o modelo base tem alto viés (ex.: *decision stumps*, árvores rasas).

- **AdaBoost** (*Adaptive Boosting*): ajusta os pesos das amostras de treino a cada iteração, priorizando exemplos mal classificados.
- **Gradient Boosting**: treina cada novo modelo para corrigir os **resíduos** (gradiente do erro) do ensemble acumulado, usando descida de gradiente no espaço funcional.

## Dataset: Optical Recognition of Handwritten Digits

O dataset **digits** (disponível no scikit-learn) contém imagens 8×8 pixels de dígitos manuscritos (0–9), representadas como vetores de 64 features (intensidade de cinza de cada pixel). É um problema de classificação multiclasse com 10 classes, com alta dimensionalidade que favorece variância em árvores — tornando-o um bom banco de testes para as duas famílias de ensemble.


## Validação Cruzada Aninhada (*Nested Cross-Validation*)

Ao otimizar hiperparâmetros, existe um risco de **viés otimista**: se usarmos o mesmo conjunto de dados tanto para escolher o melhor hiperparâmetro quanto para estimar o desempenho do modelo, a estimativa ficará inflada — o modelo foi selecionado justamente por ter se saído bem *naqueles* dados.

A **validação cruzada aninhada** resolve isso com dois loops de CV aninhados:

```
Dataset (sem o teste holdout)
│
└── Loop EXTERNO (3 folds) — estima o desempenho generalizado
    │
    │   Para cada fold externo:
    │   ├── Treino externo  →  Loop INTERNO (3 folds / GridSearchCV)
    │   │                      └── seleciona o melhor n_estimators
    │   └── Validação externa → avalia o modelo com o n_estimators escolhido
    │                            (nunca visto durante a seleção interna)
    │
    └── Média das 3 acurácias externas = estimativa não-viesada de desempenho
```

O loop interno (**3 folds**) é o `GridSearchCV`: encontra o melhor `n_estimators` dentro do treino externo.  
O loop externo (**3 folds**) avalia o modelo resultante em dados que o loop interno nunca viu.

> **Papel do conjunto de teste holdout:** após toda a nested CV, retreinamos o modelo final com o `n_estimators` escolhido (determinado pela curva média dos folds internos) em **todos** os dados de desenvolvimento, e avaliamos **uma única vez** no conjunto de teste — este nunca participa de nenhum fold.


---
# Parte 1 — Configuração do Ambiente

## 1.1 Importação de Bibliotecas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100


## 1.2 Carregamento e Visualização do Dataset


In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f'Número de amostras : {X.shape[0]}')
print(f'Número de features : {X.shape[1]}  (pixels 8x8 linearizados)')
print(f'Classes            : {np.unique(y)}  ({len(np.unique(y))} dígitos)')
print()
for cls in np.unique(y):
    print(f'  Dígito {cls}: {(y == cls).sum()} amostras')


In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    for row, idx_pos in enumerate([0, 1]):
        idx = np.where(y == digit)[0][idx_pos]
        axes[row, digit].imshow(digits.images[idx], cmap='gray_r')
        if row == 0:
            axes[row, digit].set_title(f'Dígito {digit}', fontsize=9)
        axes[row, digit].axis('off')
plt.suptitle('Exemplos de imagens do dataset Digits (cada coluna = um dígito)', y=1.02)
plt.tight_layout()
plt.show()


---
# Parte 2 — Divisão dos Dados

## 2.1 Estratégia com Nested Cross-Validation

Com nested CV, a divisão é mais simples do que com hold-out de validação:

```
Dataset completo (1797 amostras)
      │
      ├─── Conjunto de Desenvolvimento (80%) ── submetido à nested CV
      │         ├── Loop externo (3 folds): estima desempenho
      │         └── Loop interno (3 folds / GridSearchCV): seleciona n_estimators
      │
      └─── Conjunto de Teste Holdout (20%) ── tocado UMA ÚNICA VEZ, no final
```

> **Não há mais um conjunto de validação fixo separado**, pois os folds externos já cumprem esse papel de forma mais robusta e sem desperdiçar dados.

A divisão inicial é **estratificada** (`stratify=y`) para preservar a proporção das 10 classes.


In [ ]:
# Separar apenas o conjunto de teste holdout — o resto vai para nested CV
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Desenvolvimento : {X_dev.shape[0]} amostras ({X_dev.shape[0]/len(X)*100:.0f}%)')
print(f'Teste holdout   : {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')
print()

# Verificar estratificação
for nome, yy in [('Desenvolvimento', y_dev), ('Teste', y_test)]:
    uniq, counts = np.unique(yy, return_counts=True)
    pcts = counts / len(yy) * 100
    print(f'{nome}: ' + '  '.join(f'{c}:{p:.1f}%' for c, p in zip(uniq, pcts)))


## 2.2 Configuração dos Folds e da Grade de Hiperparâmetros

Definimos os dois níveis de CV e o único hiperparâmetro a otimizar: `n_estimators`.


In [ ]:
# ── Configuração da nested CV ─────────────────────────────────────────────────
N_OUTER = 3   # folds externos: estimam o desempenho generalizado
N_INNER = 3   # folds internos: selecionam n_estimators (GridSearchCV)

outer_cv = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=SEED)

# Grade de busca: apenas n_estimators
n_estimators_range = [5, 10, 25, 50, 100, 200]

print(f'Loop externo : {N_OUTER} folds estratificados')
print(f'Loop interno : {N_INNER} folds estratificados (GridSearchCV)')
print(f'n_estimators a testar: {n_estimators_range}')
print(f'Total de fits por modelo: {N_OUTER} folds externos × '
      f'{len(n_estimators_range)} valores × {N_INNER} folds internos '
      f'= {N_OUTER * len(n_estimators_range) * N_INNER} treinos')


## 2.3 Função auxiliar de Nested CV

Para evitar repetição de código, encapsulamos o loop de nested CV em uma função reutilizável. Ela retorna:
- `outer_scores`: acurácia de cada fold externo (estimativa não-viesada)
- `best_ns_per_fold`: o `n_estimators` escolhido pelo GridSearchCV em cada fold externo
- `avg_val_curve` / `avg_train_curve`: curvas de acurácia × `n_estimators` médias dos folds internos
- `final_n`: `n_estimators` a usar no modelo final (argmax da curva média de validação)


In [ ]:
def nested_cv(estimator, param_grid, X, y, outer_cv, inner_cv, n_range):
    """
    Executa nested cross-validation para seleção de hiperparâmetro e
    estimativa não-viesada de desempenho.

    Parâmetros
    ----------
    estimator  : estimador scikit-learn configurado (sem n_estimators)
    param_grid : dicionário para GridSearchCV, ex.: {'n_estimators': [5, 10, ...]}
    X, y       : dados de desenvolvimento (SEM o conjunto de teste holdout)
    outer_cv   : StratifiedKFold para o loop externo
    inner_cv   : StratifiedKFold para o loop interno (GridSearchCV)
    n_range    : lista de valores de n_estimators testados

    Retorna
    -------
    outer_scores    : list[float] — acurácia em cada fold externo
    best_ns_per_fold: list[int]   — melhor n_estimators por fold externo
    avg_val_curve   : np.array   — curva val_cv média (shape = len(n_range))
    avg_train_curve : np.array   — curva train_cv média
    final_n         : int        — n_estimators recomendado para o modelo final
    """
    outer_scores     = []
    best_ns_per_fold = []
    val_curves       = []
    train_curves     = []

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # ── Loop interno: GridSearchCV com N_INNER folds ──────────────────
        grid = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            cv=inner_cv,
            scoring='accuracy',
            return_train_score=True,
            n_jobs=-1
        )
        grid.fit(X_tr, y_tr)

        # Melhor n_estimators neste fold externo
        best_n = grid.best_params_['n_estimators']
        best_ns_per_fold.append(best_n)

        # ── Loop externo: avaliar o melhor modelo na validação externa ────
        # O best_estimator_ já foi retreinado em X_tr pelo GridSearchCV
        acc_val = accuracy_score(y_val, grid.best_estimator_.predict(X_val))
        outer_scores.append(acc_val)

        # Guardar curvas internas para análise posterior
        val_curves.append(grid.cv_results_['mean_test_score'])
        train_curves.append(grid.cv_results_['mean_train_score'])

        print(f'  Fold externo {fold_idx + 1}: '
              f'melhor n_estimators = {best_n:>3d}, '
              f'acurácia validação externa = {acc_val:.4f}')

    # Médias das curvas sobre os folds externos
    avg_val_curve   = np.mean(val_curves,   axis=0)
    avg_train_curve = np.mean(train_curves, axis=0)

    # n_estimators do modelo final = argmax da curva de validação média
    final_n = n_range[int(np.argmax(avg_val_curve))]

    return outer_scores, best_ns_per_fold, avg_val_curve, avg_train_curve, final_n


---
# Parte 3 — Modelos de Referência: Árvores de Decisão

Antes de aplicar métodos de ensemble, estabelecemos dois **modelos de referência** (*baseline*):

1. **Árvore sem poda** (`max_depth=None`): árvore profunda, potencial de **alta variância** (overfitting). Modelo base natural para Bagging e Random Forests.
2. **Árvore com poda** (`max_depth=1`, *decision stump*): árvore rasa, potencial de **alto viés** (underfitting). Modelo base típico para AdaBoost (aprendiz fraco).

Para os baselines, aplicamos apenas o **loop externo** da nested CV (sem seleção de hiperparâmetros): obtemos 3 acurácias externas cuja média é a estimativa de desempenho esperado.

> **Conceito importante:**
> - Alta variância → grande variação entre os folds externos e gap treino >> validação
> - Alto viés → acurácias baixas e estáveis nos folds externos


In [ ]:
# ── DT sem poda: cross_val_score com o loop externo ─────────────────────────
dt_sem_poda = DecisionTreeClassifier(random_state=SEED)
dt_sp_outer_scores = cross_val_score(
    dt_sem_poda, X_dev, y_dev, cv=outer_cv, scoring='accuracy'
)

print('=== Árvore de Decisão SEM poda (max_depth=None) ===')
for i, s in enumerate(dt_sp_outer_scores):
    print(f'  Fold externo {i+1}: {s:.4f}')
print(f'  Média ± Desvio padrão: {dt_sp_outer_scores.mean():.4f} ± {dt_sp_outer_scores.std():.4f}')
print()

# ── DT com poda (stump): idem ─────────────────────────────────────────────────
dt_com_poda = DecisionTreeClassifier(max_depth=1, random_state=SEED)
dt_cp_outer_scores = cross_val_score(
    dt_com_poda, X_dev, y_dev, cv=outer_cv, scoring='accuracy'
)

print('=== Árvore de Decisão COM poda (max_depth=1 — decision stump) ===')
for i, s in enumerate(dt_cp_outer_scores):
    print(f'  Fold externo {i+1}: {s:.4f}')
print(f'  Média ± Desvio padrão: {dt_cp_outer_scores.mean():.4f} ± {dt_cp_outer_scores.std():.4f}')


### Interpretação esperada

- A **árvore sem poda** deve apresentar acurácias mais altas mas com **maior variação entre folds** — reflexo da instabilidade causada pela alta variância.
- A **árvore com poda** deve apresentar acurácias muito baixas e estáveis — reflexo do alto viés.

Essas duas configurações motivam as duas famílias de ensemble:
- Bagging e Random Forests → reduzem a **variância** da árvore sem poda
- AdaBoost e Gradient Boosting → reduzem o **viés** da árvore com poda


---
# Parte 4 — Bagging

## 4.1 Como funciona o Bagging?

O **Bagging** (*Bootstrap AGGregatiNG*) reduz a variância do modelo base por diversificação:

1. São criadas $T$ amostras de treino via *bootstrap* (amostragem com reposição do conjunto original).
2. Um modelo base é treinado independentemente em cada amostra.
3. A predição final agrega as $T$ predições por **votação majoritária** (classificação).

Como cada árvore é treinada em um conjunto ligeiramente diferente, elas cometem erros em regiões distintas do espaço de features. A votação cancela parte desses erros, reduzindo a variância total.

## 4.2 Nested CV — seleção de n_estimators e estimativa de desempenho

O loop interno (GridSearchCV com 3 folds) encontra o melhor `n_estimators` dentro de cada treino externo. O loop externo avalia o resultado em dados que o loop interno nunca viu.


In [ ]:
print('=== BAGGING — Nested Cross-Validation ===')
print(f'Grid de busca: n_estimators = {n_estimators_range}')
print()

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    random_state=SEED,
    n_jobs=-1
)

bag_outer, bag_best_ns, bag_val_curve, bag_train_curve, bag_final_n = nested_cv(
    estimator=bag_clf,
    param_grid={'n_estimators': n_estimators_range},
    X=X_dev, y=y_dev,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    n_range=n_estimators_range
)

print()
print(f'Acurácia por fold externo : {[round(s, 4) for s in bag_outer]}')
print(f'Média ± Desvio padrão     : {np.mean(bag_outer):.4f} ± {np.std(bag_outer):.4f}')
print(f'Melhor n_estimators (final): {bag_final_n}')


In [ ]:
# ── Curva de aprendizado: média dos 3 folds externos ─────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_estimators_range, bag_train_curve, 'o-',  label='Treino (CV interno, média)',    color='steelblue')
ax.plot(n_estimators_range, bag_val_curve,   's--', label='Validação (CV interno, média)',  color='salmon')
ax.axvline(bag_final_n, linestyle=':', color='gray', label=f'n final = {bag_final_n}')
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('Bagging — Curva de Otimização (média sobre folds externos)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Tabela: acurácia de validação por n_estimators (média dos folds externos)')
print(f'{"n_estimators":<15} {"val_cv (média)":<18} {"train_cv (média)"}')
print('-' * 52)
for n, v, t in zip(n_estimators_range, bag_val_curve, bag_train_curve):
    marker = ' ← melhor' if n == bag_final_n else ''
    print(f'{n:<15} {v:.4f}             {t:.4f}{marker}')


---
# Parte 5 — Random Forests

## 5.1 Como funciona o Random Forest?

O **Random Forest** é uma extensão do Bagging que introduz aleatoriedade adicional: ao dividir cada nó, o algoritmo considera apenas um **subconjunto aleatório de features** (em vez de todas), escolhendo a melhor dentre esse subconjunto.

| | Bagging | Random Forest |
|---|---|---|
| Bootstrap de amostras | ✓ | ✓ |
| Seleção aleatória de features | ✗ | ✓ |
| Diversidade entre árvores | Moderada | Alta |
| Tendência de variância | Reduz | Reduz ainda mais |

## 5.2 Nested CV


In [ ]:
print('=== RANDOM FOREST — Nested Cross-Validation ===')
print(f'Grid de busca: n_estimators = {n_estimators_range}')
print()

rf_clf = RandomForestClassifier(random_state=SEED, n_jobs=-1)

rf_outer, rf_best_ns, rf_val_curve, rf_train_curve, rf_final_n = nested_cv(
    estimator=rf_clf,
    param_grid={'n_estimators': n_estimators_range},
    X=X_dev, y=y_dev,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    n_range=n_estimators_range
)

print()
print(f'Acurácia por fold externo : {[round(s, 4) for s in rf_outer]}')
print(f'Média ± Desvio padrão     : {np.mean(rf_outer):.4f} ± {np.std(rf_outer):.4f}')
print(f'Melhor n_estimators (final): {rf_final_n}')


In [ ]:
# ── Curva comparativa: Bagging vs. Random Forest ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, t_curve, v_curve, best_n, titulo in [
    (axes[0], bag_train_curve, bag_val_curve, bag_final_n, 'Bagging'),
    (axes[1], rf_train_curve,  rf_val_curve,  rf_final_n,  'Random Forest')
]:
    ax.plot(n_estimators_range, t_curve, 'o-',  label='Treino CV (média)',    color='steelblue')
    ax.plot(n_estimators_range, v_curve, 's--', label='Validação CV (média)', color='salmon')
    ax.axvline(best_n, linestyle=':', color='gray', label=f'n final = {best_n}')
    ax.set_xlabel('n_estimators', fontsize=12)
    ax.set_ylabel('Acurácia', fontsize=12)
    ax.set_title(titulo, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Acurácia × Número de Estimadores — Métodos Paralelos\n(curvas médias da nested CV)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 5.3 Importância das Features

Uma vantagem do Random Forest é fornecer uma medida de **importância das features**, calculada com base na redução média do índice de Gini em todas as divisões que utilizam cada feature. No contexto do dataset digits, isso nos permite visualizar quais **regiões dos dígitos** (pixels) são mais discriminativas.

Como o Random Forest precisa estar treinado para extrair as importâncias, treinamos aqui um modelo temporário com o `n_final` selecionado, usando todos os dados de desenvolvimento.


In [ ]:
# Treinar RF temporário para visualizar importâncias (usa todos os dados de dev)
rf_temp = RandomForestClassifier(n_estimators=rf_final_n, random_state=SEED, n_jobs=-1)
rf_temp.fit(X_dev, y_dev)

importances = rf_temp.feature_importances_.reshape(8, 8)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(importances, cmap='hot', interpolation='nearest')
plt.colorbar(im, ax=ax, label='Importância (redução de Gini)')
ax.set_title(f'Importância das Features — RF (n={rf_final_n})\n(pixels 8×8 do dígito)', fontsize=12)
ax.set_xticks(range(8)); ax.set_yticks(range(8))
ax.set_xlabel('Coluna do pixel'); ax.set_ylabel('Linha do pixel')
plt.tight_layout()
plt.show()
print('Pixels mais escuros = menos importantes | Pixels mais quentes = mais importantes')
print('Os pixels das bordas tendem a ser menos discriminativos.')


---
# Parte 6 — AdaBoost

## 6.1 Como funciona o AdaBoost?

O **AdaBoost** treina os modelos base **sequencialmente**. A cada iteração $t$:

1. Um aprendiz fraco $h_t(x)$ é treinado com pesos nas amostras: exemplos mal classificados recebem **peso maior**.
2. O peso do aprendiz fraco $\\alpha_t = \\frac{1}{2}\\ln\\frac{1-\\varepsilon_t}{\\varepsilon_t}$ depende do seu erro ponderado $\\varepsilon_t$.
3. Os pesos das amostras são atualizados: mal classificadas ficam mais pesadas.

$$H(x) = \\text{sign}\\left(\\sum_{t=1}^{T} \\alpha_t h_t(x)\\right)$$

**Por que usar aprendiz fraco (árvore com poda)?**  
O AdaBoost busca corrigir o **viés** iterativamente. Se o aprendiz base já for muito complexo (alta variância), ele memoriza os dados com peso aumentado e causa overfitting rapidamente. Aprendizes fracos (stumps) são estáveis e permitem o boosting corrigir o viés de forma controlada.

## 6.2 Nested CV


In [ ]:
print('=== ADABOOST — Nested Cross-Validation ===')
print(f'Modelo base: DecisionTreeClassifier(max_depth=1) — decision stump')
print(f'Grid de busca: n_estimators = {n_estimators_range}')
print()

ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED),
    random_state=SEED
)

ada_outer, ada_best_ns, ada_val_curve, ada_train_curve, ada_final_n = nested_cv(
    estimator=ada_clf,
    param_grid={'n_estimators': n_estimators_range},
    X=X_dev, y=y_dev,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    n_range=n_estimators_range
)

print()
print(f'Acurácia por fold externo : {[round(s, 4) for s in ada_outer]}')
print(f'Média ± Desvio padrão     : {np.mean(ada_outer):.4f} ± {np.std(ada_outer):.4f}')
print(f'Melhor n_estimators (final): {ada_final_n}')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_estimators_range, ada_train_curve, 'o-',  label='Treino CV (média)',    color='steelblue')
ax.plot(n_estimators_range, ada_val_curve,   's--', label='Validação CV (média)', color='salmon')
ax.axvline(ada_final_n, linestyle=':', color='gray', label=f'n final = {ada_final_n}')
ax.set_xlabel('n_estimators', fontsize=12)
ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('AdaBoost — Curva Otimização (média sobre folds externos)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Compare com o stump isolado (média da nested CV externa):')
print(f'  DT stump: {dt_cp_outer_scores.mean():.4f}  →  AdaBoost: {np.mean(ada_outer):.4f}')
print(f'  Ganho: {np.mean(ada_outer) - dt_cp_outer_scores.mean():.4f}')


---
# Parte 7 — Gradient Boosting

## 7.1 Como funciona o Gradient Boosting?

O **Gradient Boosting** generaliza o boosting: cada novo modelo base é treinado para **corrigir os resíduos** do ensemble acumulado.

$$F_t(x) = F_{t-1}(x) + \\eta \\cdot h_t(x)$$

onde $h_t$ minimiza o gradiente da função de perda e $\\eta$ (*learning rate*) controla a contribuição de cada novo modelo.

| | AdaBoost | Gradient Boosting |
|---|---|---|
| Mecanismo de foco | Pesos nas amostras | Resíduos (gradiente da perda) |
| Flexibilidade de perda | Limitada | Qualquer função diferenciável |
| Robustez a outliers | Mais sensível | Mais robusto |
| Profundidade do modelo base | 1 (stump) | 3–8 (árvores rasas, `max_depth=3` default) |

## 7.2 Nested CV


In [ ]:
print('=== GRADIENT BOOSTING — Nested Cross-Validation ===')
print(f'Modelo base interno: árvore rasa (max_depth=3, default)')
print(f'Grid de busca: n_estimators = {n_estimators_range}')
print()

gb_clf = GradientBoostingClassifier(random_state=SEED)

gb_outer, gb_best_ns, gb_val_curve, gb_train_curve, gb_final_n = nested_cv(
    estimator=gb_clf,
    param_grid={'n_estimators': n_estimators_range},
    X=X_dev, y=y_dev,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    n_range=n_estimators_range
)

print()
print(f'Acurácia por fold externo : {[round(s, 4) for s in gb_outer]}')
print(f'Média ± Desvio padrão     : {np.mean(gb_outer):.4f} ± {np.std(gb_outer):.4f}')
print(f'Melhor n_estimators (final): {gb_final_n}')


In [ ]:
# ── Curva comparativa: AdaBoost vs. Gradient Boosting ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, t_curve, v_curve, best_n, titulo in [
    (axes[0], ada_train_curve, ada_val_curve, ada_final_n, 'AdaBoost'),
    (axes[1], gb_train_curve,  gb_val_curve,  gb_final_n,  'Gradient Boosting')
]:
    ax.plot(n_estimators_range, t_curve, 'o-',  label='Treino CV (média)',    color='steelblue')
    ax.plot(n_estimators_range, v_curve, 's--', label='Validação CV (média)', color='salmon')
    ax.axvline(best_n, linestyle=':', color='gray', label=f'n final = {best_n}')
    ax.set_xlabel('n_estimators', fontsize=12)
    ax.set_ylabel('Acurácia', fontsize=12)
    ax.set_title(titulo, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Acurácia × Número de Estimadores — Métodos Sequenciais\n(curvas médias da nested CV)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


---
# Parte 8 — Comparação Final dos Modelos

## 8.1 Resumo das Estimativas da Nested CV

Antes de tocar o conjunto de teste, comparamos as **estimativas não-viesadas de desempenho** obtidas pela nested CV (folds externos) para cada modelo.

> As estimativas da nested CV são mais confiáveis do que uma simples acurácia de validação, pois o `n_estimators` foi selecionado nos dados internos e avaliado em dados externos independentes.


In [ ]:
resultados_ncv = {
    'DT sem poda'     : {'scores': dt_sp_outer_scores,  'final_n': None},
    'DT com poda'     : {'scores': dt_cp_outer_scores,  'final_n': None},
    'Bagging'         : {'scores': np.array(bag_outer),            'final_n': bag_final_n},
    'Random Forest'   : {'scores': np.array(rf_outer),             'final_n': rf_final_n},
    'AdaBoost'        : {'scores': np.array(ada_outer),            'final_n': ada_final_n},
    'Gradient Boosting': {'scores': np.array(gb_outer),            'final_n': gb_final_n},
}

print(f'{'Modelo':<22} {'Fold 1':>8} {'Fold 2':>8} {'Fold 3':>8} {'Média':>8} {'± DP':>8} {'n final':>9}')
print('-' * 77)
for nome, d in resultados_ncv.items():
    s = d['scores']
    n_str = str(d['final_n']) if d['final_n'] is not None else '—'
    print(f'{nome:<22} {s[0]:>8.4f} {s[1]:>8.4f} {s[2]:>8.4f} '
          f'{s.mean():>8.4f} {s.std():>8.4f} {n_str:>9}')

In [ ]:
# ── Gráfico: estimativas nested CV com barras de erro ────────────────────────
nomes_plot = list(resultados_ncv.keys())
medias = [resultados_ncv[n]['scores'].mean() for n in nomes_plot]
stds   = [resultados_ncv[n]['scores'].std()  for n in nomes_plot]
cores  = ['#e74c3c', '#c0392b', '#3498db', '#2980b9', '#f39c12', '#27ae60']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(nomes_plot, medias, color=cores, alpha=0.85, edgecolor='k', linewidth=0.7)
ax.errorbar(nomes_plot, medias, yerr=stds, fmt='none', color='black',
            capsize=6, capthick=1.5, elinewidth=1.5)

for bar, m, s in zip(bars, medias, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds) + 0.005,
            f'{m:.4f}\n±{s:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Acurácia (nested CV — folds externos)', fontsize=12)
ax.set_title('Estimativa de Desempenho — Nested Cross-Validation (3 folds externos)', fontsize=13)
ax.set_ylim(0.0, 1.12)
ax.set_xticklabels(nomes_plot, rotation=15, ha='right', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 8.2 Avaliação Final no Conjunto de Teste Holdout

> ⚠️ **Regra de ouro:** o conjunto de teste **só pode ser utilizado uma vez**, após todos os `n_estimators` terem sido selecionados pela nested CV. Tocá-lo antes constituiria *data leakage*.

**Protocolo:**
1. Retreinar cada modelo com o `n_estimators` selecionado pela nested CV em **todos** os dados de desenvolvimento (`X_dev`).
2. Avaliar **uma única vez** no conjunto de teste holdout.

A acurácia no teste é o valor reportável — e a estimativa da nested CV permite saber se há gap entre o desempenho esperado (nested CV) e o real (teste).


In [ ]:
# ── Retreinar em X_dev completo e avaliar em X_test ─────────────────────────
modelos_finais = {
    'DT sem poda'        : DecisionTreeClassifier(random_state=SEED),
    'DT com poda'        : DecisionTreeClassifier(max_depth=1, random_state=SEED),
    f'Bagging (n={bag_final_n})'          : BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED),
        n_estimators=bag_final_n, random_state=SEED, n_jobs=-1),
    f'Random Forest (n={rf_final_n})'     : RandomForestClassifier(
        n_estimators=rf_final_n, random_state=SEED, n_jobs=-1),
    f'AdaBoost (n={ada_final_n})'         : AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED),
        n_estimators=ada_final_n, random_state=SEED),
    f'Gradient Boosting (n={gb_final_n})' : GradientBoostingClassifier(
        n_estimators=gb_final_n, random_state=SEED),
}

resultados_teste = {}
print(f'{'Modelo':<35} {'Treino':>8} {'Teste':>8} {'Gap':>8}')
print('-' * 63)
for nome, modelo in modelos_finais.items():
    modelo.fit(X_dev, y_dev)   # treinar em TODOS os dados de desenvolvimento
    acc_tr  = accuracy_score(y_dev,  modelo.predict(X_dev))
    acc_tst = accuracy_score(y_test, modelo.predict(X_test))
    resultados_teste[nome] = {'Treino': acc_tr, 'Teste': acc_tst, 'Gap': acc_tr - acc_tst}
    print(f'{nome:<35} {acc_tr:>8.4f} {acc_tst:>8.4f} {acc_tr - acc_tst:>8.4f}')


In [ ]:
# ── Gráfico comparativo: nested CV vs. teste holdout ─────────────────────────
nomes_f = list(modelos_finais.keys())
acc_ncv = [resultados_ncv[k.split(' (')[0]]['scores'].mean() for k in nomes_f]
acc_tst = [resultados_teste[k]['Teste'] for k in nomes_f]

x = np.arange(len(nomes_f))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, acc_ncv, w, label='Nested CV (estimativa)', color='steelblue', alpha=0.8)
b2 = ax.bar(x + w/2, acc_tst, w, label='Teste holdout',           color='salmon',    alpha=0.8)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Acurácia', fontsize=12)
ax.set_title('Estimativa Nested CV vs. Acurácia no Teste Holdout', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(nomes_f, rotation=18, ha='right', fontsize=9)
ax.set_ylim(0.0, 1.10)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Um bom acordo entre nested CV e teste holdout indica que a estimativa não está viesada.')


## 8.3 Análise de Viés × Variância

Comparamos o **gap treino-teste** (proxy de variância) com a **acurácia no teste** para cada modelo. Espera-se:

- **Alto gap + alta acurácia no treino** → alta variância (overfitting).
- **Baixo gap + baixa acurácia** → alto viés (underfitting).
- **Baixo gap + alta acurácia** → boa generalização.


In [ ]:
gaps      = [resultados_teste[n]['Gap']   for n in nomes_f]
accs_tst  = [resultados_teste[n]['Teste']  for n in nomes_f]
cores6    = ['#e74c3c', '#c0392b', '#3498db', '#2980b9', '#f39c12', '#27ae60']

fig, ax = plt.subplots(figsize=(8, 5))
for nome, gap, acc, cor in zip(nomes_f, gaps, accs_tst, cores6):
    ax.scatter(gap, acc, color=cor, s=180, zorder=5, edgecolors='k', linewidths=0.8)
    ax.annotate(nome.split(' (')[0], (gap, acc),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.axvline(0.05, linestyle='--', color='gray', alpha=0.5)
ax.text(0.052, min(accs_tst) + 0.01, 'Alta variância →', color='gray', fontsize=9)
ax.set_xlabel('Gap Treino − Teste  (proxy de variância)', fontsize=12)
ax.set_ylabel('Acurácia no Teste', fontsize=12)
ax.set_title('Viés × Variância — Modelos de Ensemble', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8.4 Melhor Modelo — Matriz de Confusão


In [ ]:
melhor_nome = max(resultados_teste, key=lambda n: resultados_teste[n]['Teste'])
melhor_modelo = modelos_finais[melhor_nome]

print(f'Melhor modelo: {melhor_nome}')
print(f'Acurácia no teste: {resultados_teste[melhor_nome]["Teste"]:.4f}')
print()
print(classification_report(y_test, melhor_modelo.predict(X_test),
                             target_names=[str(d) for d in range(10)]))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_estimator(
    melhor_modelo, X_test, y_test,
    display_labels=list(range(10)),
    colorbar=True, ax=ax, cmap='Blues'
)
ax.set_title(f'Matriz de Confusão — {melhor_nome}', fontsize=12)
plt.tight_layout()
plt.show()


---
# Resumo dos Conceitos Abordados

| Método | Família | Modelo base ideal | Efeito principal | n_estimators final |
|---|---|---|---|---|
| Bagging | Paralela | Alta variância (árvore profunda) | Reduz variância | selecionado pela nested CV |
| Random Forest | Paralela | Alta variância (árvore profunda) | Reduz variância (+diversidade) | selecionado pela nested CV |
| AdaBoost | Sequencial | Alto viés (stump) | Reduz viés iterativamente | selecionado pela nested CV |
| Gradient Boosting | Sequencial | Alto viés (árvore rasa) | Reduz viés via gradiente | selecionado pela nested CV |

**Boas práticas aplicadas neste notebook:**
- **Nested cross-validation** (3 folds externos × 3 folds internos) para seleção de hiperparâmetros e estimativa de desempenho não-viesada.
- Conjunto de **teste holdout** (20%) separado antes de qualquer modelagem e tocado **uma única vez**.
- Divisão estratificada (`stratify=y`) para preservar a proporção das 10 classes em todos os subconjuntos.
- `random_state` fixo em todos os componentes para reprodutibilidade.
- Função `nested_cv` reutilizável e documentada com docstring explícita.
